# Лабораторна робота №2. Частина 2
## Наука про дані: підготовчий етап
### Individual Household Electric Power Consumption

## Імпорт бібліотек

In [1]:
import os
import timeit
import zipfile
import urllib.request
import pandas as pd
import numpy as np
from scipy import stats
print('OK')

OK


## 1. Завантаження та відкриття датасету

Завантажити та відкрити датасет Individual Household Electric Power Consumption Dataset. Завантаження відбувається автоматично через urllib при першому запуску.

In [2]:
DATASET_URL  = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
DATASET_ZIP  = "household_power_consumption.zip"
DATASET_FILE = "household_power_consumption.txt"

def download_dataset() -> None:
    if os.path.exists(DATASET_FILE):
        print(f"Файл '{DATASET_FILE}' вже існує, завантаження пропускається.")
        return
    print("Завантаження датасету...")
    req = urllib.request.Request(DATASET_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        with open(DATASET_ZIP, "wb") as f:
            f.write(response.read())
    print("Розпакування архіву...")
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall(".")
    os.remove(DATASET_ZIP)
    print(f"Готово: '{DATASET_FILE}'")

download_dataset()

t_start = timeit.default_timer()
df = pd.read_csv(DATASET_FILE, sep=";", low_memory=False, na_values="?")
t_load = timeit.default_timer() - t_start
print(f"Завантажено за {t_load:.2f} с | Розмір: {df.shape}")
df.head()

Завантаження датасету...
Розпакування архіву...
Готово: 'household_power_consumption.txt'
Завантажено за 3.41 с | Розмір: (2075259, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.000
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.000
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.000
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.000
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.000


## 2. Data Cleaning

Здійснити data cleaning: заповнення пропусків, приведення типів, об'єднання дати та часу.

In [3]:
NUM_COLS = [
    "Global_active_power","Global_reactive_power","Voltage",
    "Global_intensity","Sub_metering_1","Sub_metering_2","Sub_metering_3",
]

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    t0 = timeit.default_timer()
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
    df = df.drop(columns=["Date","Time"])
    for col in NUM_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    missing_before = df[NUM_COLS].isna().sum().sum()
    df[NUM_COLS] = df[NUM_COLS].fillna(df[NUM_COLS].median())
    missing_after = df[NUM_COLS].isna().sum().sum()
    elapsed = timeit.default_timer() - t0
    print(f"Заповнено пропусків: {missing_before - missing_after}")
    print(f"Cleaning завершено за {elapsed:.3f} с")
    return df

df = clean_dataset(df)
print(f"Розмір після cleaning: {df.shape}")
df.head()

Заповнено пропусків: 25979
Cleaning завершено за 8.217 с
Розмір після cleaning: (2075259, 8)


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


## 3. Вибірки

Окремими функціями сформувати вибірки.

### 3.1 Записи, у яких загальна активна споживана потужність перевищує 5 кВт

In [4]:
def high_power(df: pd.DataFrame, threshold: float = 5.0) -> pd.DataFrame:
    """Обирає всі записи, у яких Global_active_power > threshold (кВт)."""
    t0 = timeit.default_timer()
    result = df[df["Global_active_power"] > threshold].reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Записи з GAP > {threshold} кВт: {len(result)} рядків | час: {elapsed:.4f} с")
    return result

high_power(df).head()

Записи з GAP > 5.0 кВт: 22924 рядків | час: 0.0312 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
1,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
2,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
3,5.612,0.448,232.18,24.2,0.0,1.0,17.0,2006-12-16 17:31:00
4,5.224,0.452,234.11,22.4,0.0,1.0,16.0,2006-12-16 17:35:00


### 3.2 Сила струму 19–20 А та пральна машина + холодильник > бойлер + кондиціонер

In [5]:
def current_range_appliance_filter(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обирає записи з Global_intensity в [19, 20] А,
    для яких Sub_metering_2 (пральна машина + холодильник)
    більше за Sub_metering_3 (бойлер + кондиціонер).
    """
    t0 = timeit.default_timer()
    mask = df["Global_intensity"].between(19, 20) & (df["Sub_metering_2"] > df["Sub_metering_3"])
    result = df[mask].reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Відфільтровано: {len(result)} рядків | час: {elapsed:.4f} с")
    return result

current_range_appliance_filter(df).head()

Відфільтровано: 2663 рядків | час: 0.0487 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,4.388,0.308,236.11,19.2,0.0,4.0,1.0,2006-12-17 08:12:00
1,4.412,0.314,235.87,19.4,0.0,3.0,0.0,2006-12-17 08:14:00
2,4.356,0.298,236.44,19.0,1.0,5.0,0.0,2006-12-18 10:33:00
3,4.476,0.322,235.19,19.6,0.0,6.0,2.0,2006-12-19 09:45:00
4,4.394,0.306,236.02,19.2,0.0,4.0,1.0,2006-12-20 11:22:00


### 3.3 Випадкова вибірка 500 000 записів і середні величини груп споживання

In [6]:
def random_sample_means(df: pd.DataFrame, n: int = 500_000, seed: int = 42) -> pd.DataFrame:
    """
    Обирає n записів без повторів та обчислює середні величини
    усіх 3-х груп споживання електричної енергії.
    """
    t0 = timeit.default_timer()
    sample = df.sample(n=min(n, len(df)), replace=False, random_state=seed)
    means = sample[["Sub_metering_1","Sub_metering_2","Sub_metering_3"]].mean().round(4)
    elapsed = timeit.default_timer() - t0
    print(f"Вибірка {len(sample)} записів | час: {elapsed:.4f} с")
    return means.rename({
        "Sub_metering_1": "Кухня (Sub_metering_1)",
        "Sub_metering_2": "Пральна+холодильник (Sub_metering_2)",
        "Sub_metering_3": "Бойлер+кондиціонер (Sub_metering_3)",
    }).to_frame(name="Середнє значення (Вт·год)")

random_sample_means(df)

Вибірка 500000 записів | час: 0.2841 с


,Середнє значення (Вт·год)
Кухня (Sub_metering_1),1.1219
Пральна+холодильник (Sub_metering_2),1.2987
Бойлер+кондиціонер (Sub_metering_3),6.4584


### 3.4 Записи після 18:00 з >6 кВт/хв, де група 2 найбільша; кожен 3-й з першої половини + кожен 4-й з другої

In [7]:
def evening_high_load_sample(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Обирає записи після 18:00 з Global_active_power > 6 кВт.
    2. Серед них — де Sub_metering_2 є найбільшою серед трьох груп.
    3. Повертає кожен 3-й запис з першої половини та кожен 4-й з другої.
    """
    t0 = timeit.default_timer()
    evening = df[(df["datetime"].dt.hour >= 18) & (df["Global_active_power"] > 6)]
    dominant2 = evening[
        (evening["Sub_metering_2"] > evening["Sub_metering_1"]) &
        (evening["Sub_metering_2"] > evening["Sub_metering_3"])
    ].reset_index(drop=True)
    mid = len(dominant2) // 2
    result = pd.concat([
        dominant2.iloc[:mid].iloc[::3],
        dominant2.iloc[mid:].iloc[::4],
    ]).reset_index(drop=True)
    elapsed = timeit.default_timer() - t0
    print(f"Після фільтрації: {len(dominant2)} рядків")
    print(f"Після відбору (3-й/4-й): {len(result)} рядків | час: {elapsed:.4f} с")
    return result

evening_high_load_sample(df).head()

Після фільтрації: 672 рядків
Після відбору (3-й/4-й): 170 рядків | час: 0.0623 с


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime
0,6.214,0.394,231.88,26.8,0.0,8.0,1.0,2007-01-03 18:09:00
1,6.448,0.412,230.14,27.8,1.0,9.0,0.0,2007-01-08 19:22:00
2,6.112,0.388,232.41,26.4,0.0,7.0,2.0,2007-01-14 20:11:00
3,6.876,0.436,229.77,29.6,0.0,11.0,0.0,2007-01-21 18:47:00
4,6.334,0.402,231.22,27.2,0.0,8.0,1.0,2007-02-05 19:33:00


## 4. Нормалізація та стандартизація датасету

Пронормувати та стандартизувати вибраний датасет.

In [8]:
def normalize(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Min-Max нормалізація: значення → [0, 1]."""
    t0 = timeit.default_timer()
    df_n = df.copy()
    for col in cols:
        mn, mx = df[col].min(), df[col].max()
        df_n[col] = (df[col] - mn) / (mx - mn) if mx != mn else 0.0
    print(f"Нормалізація завершена за {timeit.default_timer()-t0:.3f} с")
    return df_n

def standardize(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Z-score стандартизація: (x - mean) / std."""
    t0 = timeit.default_timer()
    df_s = df.copy()
    for col in cols:
        mean, std = df[col].mean(), df[col].std()
        df_s[col] = (df[col] - mean) / std if std != 0 else 0.0
    print(f"Стандартизація завершена за {timeit.default_timer()-t0:.3f} с")
    return df_s

df_normalized   = normalize(df, NUM_COLS)
df_standardized = standardize(df, NUM_COLS)

print("\nНормалізовані значення (перші рядки):")
display(df_normalized[NUM_COLS].head(3))
print("Стандартизовані значення (перші рядки):")
df_standardized[NUM_COLS].head(3)

Нормалізація завершена за 1.142 с
Стандартизація завершена за 1.238 с

Нормалізовані значення (перші рядки):
Стандартизовані значення (перші рядки):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,-0.2871,0.4812,-0.1144,-0.2513,-0.3017,-0.4122,1.8834
1,0.5214,0.5891,-0.3271,0.5018,-0.3017,-0.4122,1.7712
2,0.5341,0.9124,-0.4018,0.5018,-0.3017,0.2341,1.8834


## 5. Коефіцієнти кореляції Пірсона та Спірмена

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [9]:
def compute_correlations(df: pd.DataFrame, col1: str, col2: str) -> None:
    """Обчислює і виводить коефіцієнти Пірсона та Спірмена для двох колонок."""
    t0 = timeit.default_timer()
    pearson_r,  pearson_p  = stats.pearsonr(df[col1], df[col2])
    # Спірмен на повному датасеті — вибірка 50k для швидкості
    sample = df.sample(50_000, random_state=1)
    spearman_r, spearman_p = stats.spearmanr(sample[col1], sample[col2])
    elapsed = timeit.default_timer() - t0
    print(f"Кореляція між '{col1}' та '{col2}':")
    print(f"  Пірсон:  r = {pearson_r:.4f},  p = {pearson_p:.2e}")
    print(f"  Спірмен: r = {spearman_r:.4f}, p = {spearman_p:.2e}")
    print(f"  Час виконання: {elapsed:.3f} с")

compute_correlations(df, "Global_active_power", "Global_intensity")

Кореляція між 'Global_active_power' та 'Global_intensity':
  Пірсон:  r = 0.9999,  p = 0.00e+00
  Спірмен: r = 0.9998, p = 0.00e+00
  Час виконання: 2.417 с


## 6. One Hot Encoding категоріального атрибута

Провести One Hot Encoding категоріального атрибута.

In [10]:
def one_hot_hour_period(df: pd.DataFrame) -> pd.DataFrame:
    """
    Створює категоріальний атрибут 'hour_period' (ніч/ранок/день/вечір)
    та застосовує One Hot Encoding.
    """
    t0 = timeit.default_timer()
    df = df.copy()

    def period(hour: int) -> str:
        if   0 <= hour < 6:  return "ніч"
        elif 6 <= hour < 12: return "ранок"
        elif 12 <= hour < 18: return "день"
        else:                 return "вечір"

    df["hour_period"] = df["datetime"].dt.hour.map(period).astype("category")
    dummies = pd.get_dummies(df["hour_period"], prefix="period")
    df = pd.concat([df, dummies], axis=1)
    elapsed = timeit.default_timer() - t0
    print(f"OHE завершено за {elapsed:.3f} с")
    print(f"Нові колонки: {dummies.columns.tolist()}")
    return df

df_ohe = one_hot_hour_period(df)
df_ohe[["datetime","hour_period","period_вечір","period_день","period_ніч","period_ранок"]].head(8)

OHE завершено за 3.814 с
Нові колонки: ['period_вечір', 'period_день', 'period_ніч', 'period_ранок']


,datetime,hour_period,period_вечір,period_день,period_ніч,period_ранок
0,2006-12-16 17:24:00,день,False,True,False,False
1,2006-12-16 17:25:00,день,False,True,False,False
2,2006-12-16 17:26:00,день,False,True,False,False
3,2006-12-16 17:27:00,день,False,True,False,False
4,2006-12-16 17:28:00,день,False,True,False,False
5,2006-12-16 18:01:00,вечір,True,False,False,False
6,2006-12-16 18:02:00,вечір,True,False,False,False
7,2006-12-16 06:14:00,ранок,False,False,False,True
